# Where along the PT does the injury response sit, and does it move? (notebook 114)

**The question.** Notebook 94 fitted, for every gene and every Kirita time point, three segment contrasts of the
IRI response: S3−S1 (which it called "location"), S3−S2 and S2−S1. It read only the first, at one time. The rest
is a committed position × time grid in 18 mice that has never been looked at.

**The frozen protocol** is `docs/results/pt-injury-geography-over-time.md` (commit e22a88d).
- W1: is the acute response S3-weighted?
- W2: which boundary carries the gradient, S2/S3 or S1/S2?
- W3: does the geography outlast the magnitude?
- W4: do up and down programs differ?
- W5: does our own position statistic match, and at which time?
- Controls: housekeeping (C1), the notebook 112 clean filters (C2), the balanced relabelings (C3).

**Sections.** 1 setup; 2 the grid and the pool; 3 the sets; 4 matched-null z; 5 W1 and W2; 6 W3; 7 W4; 8 W5;
9 controls; 10 verdicts; 11 figures; 12 reading and run record.

## 1 · Setup and inputs

In [ ]:
import argparse
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
T94 = results_root / 'pt_external_iri' / 'tables'
T91 = results_root / 'pt_positional_decomposition' / 'tables'
T112 = results_root / 'pt_gene_atlas' / 'tables'
T76 = results_root / 'pt_package_aki' / 'tables'
TPROG = results_root / 'pt_known_injury_programs' / 'tables'
T68 = results_root / 'pt_species_level_control' / 'tables'
output = results_root / 'pt_injury_geography'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '114.geography.1'
SEED = 114
DRAWS, N_DECILES, Z_CUT = 2000, 10, 2.0
L91_CUT, M91_CUT = .5, .4
TIMES = ['4hours', '12hours', '2days', '14days', '6weeks']
TIME_LABEL = dict(zip(TIMES, ['4 h', '12 h', '2 d', '14 d', '6 wk']))
CONTRASTS = ['level', 'location', 'S3-S2', 'S2-S1']
for path in (T94 / 'external_genes.csv.gz', T94 / 'universe.csv', T91 / 'i_genes.csv.gz',
             T112 / 'aki_atlas.csv.gz', T76 / 'gene_sets.json', TPROG / 'programs_members.csv',
             TPROG / 'programs.csv', T68 / 'gene_level.csv'):
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from IPython.display import display

from pseudospace.external_replication import matched_means

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
BLUE, ORANGE, GREEN, PURPLE, GREY = '#0072B2', '#D55E00', '#009E73', '#CC79A7', '#999999'
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_rows', 100)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


guard_log = []

## 2 · The position × time grid and the matching pool

In [ ]:
EXT = pd.read_csv(T94 / 'external_genes.csv.gz', header=[0, 1], index_col=0)
EXT.index = EXT.index.astype(str)
g91 = pd.read_csv(T91 / 'i_genes.csv.gz').set_index('gene')
g91 = g91[~g91.index.duplicated()]
POOL = EXT.index.intersection(g91.index)
guard_log.append(f'G1: Kirita universe {len(EXT):,} genes; pool shared with notebook 91 {len(POOL):,}')

COLS = [(t, f'{c} coefficient') for t in TIMES for c in CONTRASTS]
ASSIGNED = [(t, f'{c} coefficient') for t in TIMES
            for c in ['level', 'location (assigned)', 'S3-S2', 'S2-S1']]
VAL = EXT.loc[POOL, COLS].astype(float)
VAL_A = EXT.loc[POOL, ASSIGNED].astype(float)
NAMES = [f'{TIME_LABEL[t]} {c}' for t, c in [(t, c) for t in TIMES for c in CONTRASTS]]
VAL.columns = NAMES
VAL_A.columns = NAMES
assert VAL.notna().all().all(), 'G2: the grid holds no missing coefficients'
guard_log.append(f'G2: {VAL.shape[1]} statistics per gene ({len(TIMES)} times x {len(CONTRASTS)} contrasts)')

expr = g91.loc[POOL, 'log2 CP10k (controls)']
STRATA = pd.Series(pd.qcut(expr.rank(method='first'), N_DECILES, labels=False).astype(int), index=POOL)
print('pool', len(POOL), '· grid', VAL.shape, '· strata', STRATA.value_counts().sort_index().to_dict())

## 3 · The sets

In [ ]:
members = pd.read_csv(TPROG / 'programs_members.csv')
programs = pd.read_csv(TPROG / 'programs.csv').set_index('program_id')
SETS, SOURCE = {}, {}
for pid, sub in members.groupby('program_id'):
    SETS[pid] = sorted(set(sub['gene'].astype(str)) & set(POOL))
    SOURCE[pid] = 'kirita (not independent)' if pid.startswith('K-') else str(programs.loc[pid, 'source'])

sets76 = json.loads((T76 / 'gene_sets.json').read_text())
u76 = set(pd.read_csv(T76 / 'universe.csv').gene.astype(str))
G2M = set(sets76['MSigDB_Hallmark_2020::G2-M Checkpoint']) & u76
E2F = set(sets76['MSigDB_Hallmark_2020::E2F Targets']) & u76
INJ = ['Havcr1', 'Lcn2', 'Krt20', 'Spp1', 'Clu', 'Vcam1', 'Krt8', 'Krt18', 'Anxa2', 'Cd74', 'Sox9', 'Icam1',
       'Timp1', 'Il18', 'Cdkn1a', 'Fn1', 'Ccl2', 'Myc']
tpt_table = pd.read_csv(T76 / 'programs_transporters.csv')
TPT = sorted(tpt_table.loc[tpt_table['in_T_PT'].astype(bool), 'mouse'].astype(str))
assert len(TPT) == 93, f'TPT is notebook 76\'s T_PT program ({len(TPT)})'
atlas = pd.read_csv(T112 / 'aki_atlas.csv.gz').set_index('gene')

OURS = {
    'INJ (injury markers)': INJ,
    'CYC (G2M + E2F)': sorted(G2M | E2F),
    'TPT (93 transporters)': TPT,
    '91 level up': g91.index[g91['level change'].astype(bool) & (g91['L'] > 0)].tolist(),
    '91 level down': g91.index[g91['level change'].astype(bool) & (g91['L'] < 0)].tolist(),
    '91 location change': g91.index[g91['location change'].astype(bool)].tolist(),
    '112 core (k >= 5)': atlas.index[(atlas['lines'] >= 5) & atlas['clean'] & atlas['supported']].tolist(),
}
for name, genes in OURS.items():
    SETS[name] = sorted(set(map(str, genes)) & set(POOL))
    SOURCE[name] = 'ours'

hk = pd.read_csv(T68 / 'gene_level.csv')
SETS['housekeeping (C1)'] = sorted(set(hk.loc[hk['housekeeping'].astype(bool), 'gene'].astype(str)) & set(POOL))
SOURCE['housekeeping (C1)'] = 'control'
SETS = {k: v for k, v in SETS.items() if len(v) >= 10}
print(pd.Series({k: len(v) for k, v in SETS.items()}).to_string())
guard_log.append(f'G3: {len(SETS)} sets with at least 10 members in the pool')

## 4 · Matched-null z for every set, time and contrast

In [ ]:
def set_z(values, sets, *, seed=SEED, draws=DRAWS):
    rows = []
    for name, genes in sets.items():
        idx = pd.Index(genes)
        obs = values.loc[idx].mean().to_numpy()
        null = matched_means(values.to_numpy(), STRATA.to_numpy(), STRATA.loc[idx].to_numpy(),
                             n=draws, seed=seed)
        mu, sd = null.mean(axis=0), null.std(axis=0, ddof=1)
        z = np.divide(obs - mu, sd, out=np.zeros_like(obs), where=sd > 0)
        for col, o, m, s, zz in zip(values.columns, obs, mu, sd, z):
            t, c = col.split(' ', 1)[0] + (' ' + col.split(' ')[1] if col.split(' ')[1] in ('h', 'd', 'wk') else ''), None
            rows.append({'set': name, 'statistic': col, 'members': len(genes), 'observed': o,
                         'null mean': m, 'null sd': s, 'z': zz})
    out = pd.DataFrame(rows)
    out['time'] = out['statistic'].str.rsplit(' ', n=1).str[0]
    out['contrast'] = out['statistic'].str.rsplit(' ', n=1).str[1]
    return out


Z = set_z(VAL, SETS)
Z['source'] = Z['set'].map(SOURCE)
Z.to_csv(tables / 'set_z.csv', index=False)
Z_A = set_z(VAL_A, SETS, seed=SEED + 1)
Z_A['source'] = Z_A['set'].map(SOURCE)
Z_A.to_csv(tables / 'set_z_assigned.csv', index=False)

wide = Z.pivot_table(index='set', columns=['contrast', 'time'], values='z')
order = [f'{TIME_LABEL[t]}' for t in TIMES]
display(wide.reindex(columns=pd.MultiIndex.from_product([CONTRASTS, order])).round(1))

## 5 · W1 · Is the acute response S3-weighted? · W2 · Which boundary?

In [ ]:
loc = Z[Z.contrast.eq('location')].pivot(index='set', columns='time', values='z')[order]
W1 = pd.DataFrame({'INJ': loc.loc['INJ (injury markers)'], 'TPT': loc.loc['TPT (93 transporters)'],
                   'CYC': loc.loc['CYC (G2M + E2F)']}).T
display(W1.round(1))
W1_HOLDS = bool((loc.loc['INJ (injury markers)', ['4 h', '12 h']] >= Z_CUT).any() and
                (loc.loc['TPT (93 transporters)', ['4 h', '12 h']] <= -Z_CUT).any())
print('W1 holds (INJ rises and TPT falls toward S3 acutely):', W1_HOLDS)

every = Z[Z.contrast.isin(['S3-S2', 'S2-S1'])]
BOUND = every.pivot_table(index=['set', 'time'], columns='contrast', values='z').reset_index()
BOUND['|S3-S2| - |S2-S1|'] = BOUND['S3-S2'].abs() - BOUND['S2-S1'].abs()
BOUND['source'] = BOUND['set'].map(SOURCE)
BOUND.to_csv(tables / 'boundary.csv', index=False)
indep = BOUND[~BOUND.source.str.startswith('kirita') & BOUND['set'].ne('housekeeping (C1)')]
acute = indep[indep.time.isin(['4 h', '12 h'])]
W2_SHARE = float((acute['|S3-S2| - |S2-S1|'] > 0).mean())
print(f'W2: S3-S2 carries more of the gradient in {W2_SHARE:.0%} of acute set-times '
      f'(median difference {acute["|S3-S2| - |S2-S1|"].median():.2f})')
WAVE = indep.pivot_table(index='set', columns='time', values='|S3-S2| - |S2-S1|')[order]
WAVE.to_csv(tables / 'boundary_over_time.csv')
print('\nwhich boundary dominates, by time (positive = S2/S3, negative = S1/S2):')
display(WAVE.round(2))
print('median over sets:', WAVE.median().round(2).to_dict())

## 6 · W3 · Does the geography outlast the magnitude?

In [ ]:
lev = Z[Z.contrast.eq('level')].pivot(index='set', columns='time', values='z')[order]
non_k = [s for s in SETS if not SOURCE[s].startswith('kirita') and s != 'housekeeping (C1)']
PERSIST = pd.DataFrame({
    'level 12 h': lev.loc[non_k, '12 h'], 'level 14 d': lev.loc[non_k, '14 d'], 'level 6 wk': lev.loc[non_k, '6 wk'],
    'location 12 h': loc.loc[non_k, '12 h'], 'location 14 d': loc.loc[non_k, '14 d'],
    'location 6 wk': loc.loc[non_k, '6 wk']})
PERSIST['level ratio 14 d'] = PERSIST['level 14 d'] / PERSIST['level 12 h']
PERSIST['location ratio 14 d'] = PERSIST['location 14 d'] / PERSIST['location 12 h']
PERSIST.to_csv(tables / 'persistence.csv')
display(PERSIST.round(2))
W3_HOLDS = bool(PERSIST['location ratio 14 d'].median() > PERSIST['level ratio 14 d'].median())
print(f"W3: median location ratio {PERSIST['location ratio 14 d'].median():.2f} against level "
      f"{PERSIST['level ratio 14 d'].median():.2f} · holds {W3_HOLDS}")

## 7 · W4 · Up against down

In [ ]:
PAIRS = [('G23-S1-up', 'G23-S1-down'), ('G23-S2-up', 'G23-S2-down'), ('G23-S3-up', 'G23-S3-down'),
         ('DX-PTS1-2-up', 'DX-PTS1-2-down'), ('DX-PTS3-TAL-up', 'DX-PTS3-TAL-down'),
         ('L23-aPT-up', 'L23-aPT-down'), ('H-AKI-UP', 'H-AKI-DN'), ('Z-IRI-UP', 'Z-IRI-DN'),
         ('91 level up', '91 level down')]
rows = []
for up, down in PAIRS:
    if up in loc.index and down in loc.index:
        for t in order:
            rows.append({'pair': up.rsplit('-', 1)[0], 'time': t, 'up': loc.loc[up, t], 'down': loc.loc[down, t]})
W4 = pd.DataFrame(rows)
W4.to_csv(tables / 'up_down.csv', index=False)
display(W4.pivot_table(index='pair', columns='time', values=['up', 'down'])
        .reindex(columns=pd.MultiIndex.from_product([['up', 'down'], order])).round(1))

## 8 · W5 · Our own geography, and at which time

In [ ]:
P_late = (g91.loc[POOL, 'Δ late'] - g91.loc[POOL, 'L']).astype(float)
rows = []
for t in TIMES:
    for c in CONTRASTS:
        rho, p = spearmanr(P_late, EXT.loc[POOL, (t, f'{c} coefficient')].astype(float))
        rows.append({'time': TIME_LABEL[t], 'contrast': c, 'rho': rho, 'p': p})
    rho_l, _ = spearmanr(g91.loc[POOL, 'L'].astype(float), EXT.loc[POOL, (t, 'level coefficient')].astype(float))
    rows.append({'time': TIME_LABEL[t], 'contrast': 'L against level (94)', 'rho': rho_l, 'p': np.nan})
ALIGN = pd.DataFrame(rows)
ALIGN.to_csv(tables / 'alignment.csv', index=False)
pivot = ALIGN.pivot(index='contrast', columns='time', values='rho')[order]
display(pivot.round(3))
BEST = {c: pivot.loc[c].idxmax() for c in pivot.index}
print('best time per contrast:', BEST)
W5_HOLDS = bool(order.index(BEST['S3-S2']) >= order.index(BEST['S2-S1']) and
                order.index(BEST['location']) >= order.index(BEST['L against level (94)']))
print('W5 holds (S3-S2 peaks no earlier than S2-S1, and position no earlier than level):', W5_HOLDS)

## 9 · Controls

In [ ]:
C1 = Z[Z.set.eq('housekeeping (C1)')].pivot(index='contrast', columns='time', values='z')[order]
display(C1.round(2))
C1_MAX = C1.abs().max(axis=1).round(2).to_dict()
C1_HOLDS = bool(C1.abs().to_numpy().max() < Z_CUT)
C1_POSITION = bool(max(C1_MAX[c] for c in ('location', 'S3-S2')) < Z_CUT)
print('C1 holds everywhere:', C1_HOLDS, '· per contrast max |z|:', C1_MAX)
print('C1 on the two S3 contrasts:', C1_POSITION)

clean = set(atlas.index[atlas['clean']])
CLEAN_SETS = {k: sorted(set(v) & clean) for k, v in SETS.items() if k in non_k or k == 'housekeeping (C1)'}
CLEAN_SETS = {k: v for k, v in CLEAN_SETS.items() if len(v) >= 10}
Z_CLEAN = set_z(VAL, CLEAN_SETS, seed=SEED + 2)
Z_CLEAN.to_csv(tables / 'clean_sensitivity.csv', index=False)
loc_clean = Z_CLEAN[Z_CLEAN.contrast.eq('location')].pivot(index='set', columns='time', values='z')[order]
C2 = pd.DataFrame({'full': loc[loc_clean.index]['12 h'] if False else loc.loc[loc_clean.index, '12 h'],
                   'clean only': loc_clean['12 h'],
                   'members kept': pd.Series({k: len(CLEAN_SETS[k]) for k in loc_clean.index})})
display(C2.round(2))
C2_HOLDS = bool(spearmanr(C2['full'], C2['clean only']).statistic > .9)
print('C2 holds (12 h location z unchanged by the clean filters, rho > 0.9):', C2_HOLDS)


def beyond_others(stat, others, cutoff, ratio=.5):
    s = pd.Series(stat).astype(float).abs()
    ok = s >= cutoff
    for o in others:
        ok &= pd.Series(o).astype(float).abs() <= ratio * s
    return ok.fillna(False)


REL_SETS = {}
for tag, other in (('R1', 'R2'), ('R2', 'R1')):
    lvl = beyond_others(g91[f'L {tag}'], [g91['L'], g91[f'L {other}']], L91_CUT)
    locm = beyond_others(g91[f'M {tag}'], [g91['M'], g91[f'M {other}']], M91_CUT)
    REL_SETS[f'91 level up {tag}'] = sorted(set(g91.index[lvl & (g91[f'L {tag}'] > 0)]) & set(POOL))
    REL_SETS[f'91 level down {tag}'] = sorted(set(g91.index[lvl & (g91[f'L {tag}'] < 0)]) & set(POOL))
    REL_SETS[f'91 location {tag}'] = sorted(set(g91.index[locm]) & set(POOL))
REL_SETS = {k: v for k, v in REL_SETS.items() if len(v) >= 10}
if REL_SETS:
    Z_REL = set_z(VAL, REL_SETS, seed=SEED + 3)
    Z_REL.to_csv(tables / 'relabeled.csv', index=False)
    display(Z_REL[Z_REL.contrast.eq('location')].pivot(index='set', columns='time', values='z')[order].round(2))
else:
    Z_REL = pd.DataFrame(columns=['set', 'contrast', 'time', 'z'])
    Z_REL.to_csv(tables / 'relabeled.csv', index=False)
print('relabeled sets with at least 10 members:', {k: len(v) for k, v in REL_SETS.items()})

In [ ]:
VERDICTS = pd.DataFrame([
    ('W1 the acute response is S3-weighted (INJ up, TPT down, |z| >= 2 at 4 h or 12 h)', W1_HOLDS),
    ('W2 S3-S2 carries more of the gradient than S2-S1 in most acute set-times', bool(W2_SHARE > .5)),
    ('W3 the geography outlasts the magnitude at 14 d', W3_HOLDS),
    ('W5 position peaks no earlier than level, and S3-S2 no earlier than S2-S1', W5_HOLDS),
    ('C1 housekeeping is flat on every contrast', C1_HOLDS),
    ('C1 housekeeping is flat on the S3 - S1 and S3 - S2 contrasts', C1_POSITION),
    ('C2 the clean filters do not change the 12 h geography', C2_HOLDS),
], columns=['verdict', 'holds'])
VERDICTS.to_csv(tables / 'verdicts.csv', index=False)
display(VERDICTS)

## 10 · Figures

In [ ]:
SHOW = ['INJ (injury markers)', 'TPT (93 transporters)', 'CYC (G2M + E2F)', '112 core (k >= 5)',
        'G23-FR', 'G21-VCAM', 'Z-IRI-UP', 'Z-IRI-DN', 'L23-aPT-up', 'H-AKI-UP', 'housekeeping (C1)']
SHOW = [s for s in SHOW if s in loc.index]
fig, ax = plt.subplots(figsize=(105 * MM, 55 * MM))
cmap = plt.get_cmap('turbo')
for i, s in enumerate(SHOW):
    colour = GREY if s == 'housekeeping (C1)' else cmap(i / max(len(SHOW) - 2, 1))
    ax.plot(range(len(order)), loc.loc[s, order], marker='o', ms=2.5, color=colour, label=s)
ax.axhline(0, color=MUTED, lw=.5)
for y in (-Z_CUT, Z_CUT):
    ax.axhline(y, color=LIGHT, lw=.5, ls='--')
ax.set_xticks(range(len(order)), order)
ax.set_ylabel('z of the S3 − S1 response')
ax.legend(ncol=2, loc='upper right', fontsize=5)
save(fig, '114a_location_trajectories')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(62 * MM, 58 * MM))
sub = Z[Z.contrast.isin(['S3-S2', 'S2-S1'])].pivot_table(index=['set', 'time'], columns='contrast',
                                                         values='z').reset_index()
early = sub.time.isin(['4 h', '12 h'])
ax.scatter(sub.loc[~early, 'S2-S1'], sub.loc[~early, 'S3-S2'], s=5, color=LIGHT, lw=0)
ax.scatter(sub.loc[early, 'S2-S1'], sub.loc[early, 'S3-S2'], s=6, color=ORANGE, lw=0)
lim = np.nanmax(np.abs(sub[['S3-S2', 'S2-S1']].to_numpy())) * 1.05
ax.plot([-lim, lim], [-lim, lim], color=LIGHT, lw=.5)
ax.axhline(0, color=MUTED, lw=.4); ax.axvline(0, color=MUTED, lw=.4)
ax.set_xlabel('z of S2 − S1'); ax.set_ylabel('z of S3 − S2'); ax.set_aspect('equal')
save(fig, '114b_boundary')
plt.show()

In [ ]:
x = np.arange(len(order))
fig, axes = plt.subplots(1, 2, figsize=(130 * MM, 48 * MM))
for ax, signed in zip(axes, (False, True)):
    f = (lambda d: d) if signed else np.abs
    rl = f(loc.loc[non_k, order]).div(f(loc.loc[non_k, '12 h']), axis=0)
    rv = f(lev.loc[non_k, order]).div(f(lev.loc[non_k, '12 h']), axis=0)
    for row in rl.to_numpy():
        ax.plot(x, row, color=LIGHT, lw=.4, zorder=1)
    ax.plot(x, rv.median(), marker='o', ms=3, color=GREY, label='magnitude (level)', zorder=3)
    ax.plot(x, rl.median(), marker='o', ms=3, color=ORANGE, label='geography (S3 − S1)', zorder=3)
    ax.axhline(1, color=MUTED, lw=.5)
    if signed:
        ax.axhline(0, color=MUTED, lw=.5, ls=':')
    ax.set_xticks(x, order); ax.set_ylim(-1.5 if signed else 0, 3)
    ax.set_title('keeping the sign' if signed else 'absolute')
axes[0].set_ylabel('z relative to 12 h'); axes[0].legend(fontsize=5.5)
save(fig, '114c_level_vs_location')
plt.show()
SIGN_FLIP = {t: float((np.sign(loc.loc[non_k, t]) != np.sign(loc.loc[non_k, '12 h'])).mean()) for t in order}
print('share of sets whose S3 - S1 sign differs from 12 h:', {k: round(v, 2) for k, v in SIGN_FLIP.items()})

In [ ]:
fig, ax = plt.subplots(figsize=(78 * MM, 48 * MM))
for c, colour in (('level', GREY), ('location', BLUE), ('S3-S2', ORANGE), ('S2-S1', GREEN),
                  ('L against level (94)', PURPLE)):
    if c in pivot.index:
        ax.plot(range(len(order)), pivot.loc[c, order], marker='o', ms=2.5, color=colour, label=c)
ax.axhline(0, color=MUTED, lw=.5)
ax.set_xticks(range(len(order)), order)
ax.set_ylabel("Spearman with our P_late"); ax.legend(fontsize=5)
save(fig, '114d_alignment')
plt.show()

## 11 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', 'The grid is notebook 94\'s per-gene coefficients, five times by four contrasts; "location" is S3 - S1 as that notebook defined it.'),
    ('reading', 'The pool is Kirita\'s universe intersected with notebook 91\'s, so every gene has a control expression for the strata.'),
    ('reading', 'The null draws 2,000 random sets with the set\'s own per-decile counts (matched_means) and z is (observed - null mean) / null sd.'),
    ('reading', 'Sets with fewer than 10 members in the pool are dropped; the five K-* programs are reported but never counted as replication.'),
    ('reading', 'TPT is the union of every notebook 76 set whose name contains "transport"; INJ is notebook 96\'s 18 markers; CYC is G2M + E2F.'),
    ('reading', 'W5 uses P_late = Delta late - L, which is notebook 94\'s own location sign for a gene.'),
    ('reading', 'C3 rebuilds the three 91-derived sets with a relabeling in the contrast\'s place (notebook 112 section C1).'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)

RUN = {'notebook': '114_pt_injury_geography', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED,
       'guards': guard_log, 'draws': DRAWS, 'pool': int(len(POOL)), 'sets': {k: len(v) for k, v in SETS.items()},
       'w1_holds': W1_HOLDS, 'w2_share': W2_SHARE, 'w3_holds': W3_HOLDS, 'w5_holds': W5_HOLDS,
       'c1_holds': C1_HOLDS, 'c1_max': C1_MAX, 'c1_position': C1_POSITION, 'c2_holds': C2_HOLDS,
       'best_time': BEST, 'sign_flip': SIGN_FLIP, 'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps({k: RUN[k] for k in ('w1_holds', 'w2_share', 'w3_holds', 'w5_holds', 'c1_holds', 'c1_max', 'c2_holds',
                                      'best_time', 'seconds')}, indent=2, default=str))